# 02 -- MetaPulsar (consistent) strategy on the real IPTA-DR2

**Presenter: Rutger van Haasteren.** Approx. **25 min** of the 1-hour slot.

The *consistent* combination strategy goes beyond a Frankenstein assembly by harmonising the timing models across PTAs: every astrophysical parameter that lives in the merged components (`astrometry`, `spindown`, `binary`, `dispersion`) collapses to a **single fitted entry**, while detector-specific JUMPs / EFACs / DMX bins keep their per-PTA suffix. The result is a single `BasePulsar` whose timing model is astrophysically self-consistent.

This notebook is the MetaPulsar half of the tutorial. It does two things:

1. **File and layout discovery** -- showcase MetaPulsar's regex-based directory walker, the canonical-name coordinate matcher, and `pta_summary`. These tools are how you go from "I have an IPTA release on disk" to "I have a `dict[pta_name -> list[par/tim entries]]` ready for `create_metapulsar`".
2. **Build a consistent MetaPulsar on real data** -- run the consistent combination on `J1853+1303` across EPTA dr2 + NANOGrav 9y, force a different reference PTA, and diff the rewritten consistent par files against the originals to make the harmonisation concrete.

## Step 0 -- Recover session state

We need `DATA_ROOT_STR` and `PULSAR_SUBSET` from `00_setup.ipynb`. If they are not in the IPython store, re-run `00_setup.ipynb`.

In [1]:
import sys
import warnings
from pathlib import Path

import loguru

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)


def quiet_loguru(level: str = "WARNING") -> None:
    loguru.logger.remove()
    loguru.logger.add(sys.stdout, level=level)


quiet_loguru()

%store -r DATA_ROOT_STR
%store -r PULSAR_SUBSET

DATA_ROOT = Path(DATA_ROOT_STR)
TARGET = PULSAR_SUBSET[0]
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"PULSAR_SUBSET  = {PULSAR_SUBSET}")
print(f"TARGET pulsar  = {TARGET}")

DATA_ROOT      = /workspaces/metapulsar/data/ipta-dr2
PULSAR_SUBSET  = ['J1853+1303', 'B1953+29']
TARGET pulsar  = J1853+1303


## Step 1 -- Layout discovery and `pta_summary`

MetaPulsar ships with a regex-based **layout discoverer** that scans a directory tree, identifies the conventions a PTA used for laying out its `.par`/`.tim` files (where they live, what suffixes they carry, which `INCLUDE` paths the tim files reference), and turns that into a `Layout` object. `combine_layouts` merges several `Layout`s into one, and `discover_files` resolves the actual files on disk.

Each PTA in IPTA-DR2 uses a different on-disk convention (EPTA's flat-per-pulsar tree, NANOGrav's release-versioned directories, PPTA's combined dr1+dr2 layout). The regex discoverer copes with all three out-of-the-box.

In [2]:
from metapulsar import (
    combine_layouts,
    discover_files,
    discover_layout,
    filter_file_data_by_pulsars,
    get_pulsar_names_from_file_data,
    pta_summary,
)

epta_layout = discover_layout(str(DATA_ROOT / "EPTA_v2.2"), name="EPTA dr2", verbose=False)
ppta_layout = discover_layout(str(DATA_ROOT / "PPTA_dr1dr2"), name="PPTA dr1dr2", verbose=False)
nanograv_layout = discover_layout(str(DATA_ROOT / "NANOGrav_9y"), name="NANOGrav 9y", verbose=False)

combined_layout = combine_layouts(epta_layout, ppta_layout, nanograv_layout)
file_data = discover_files(combined_layout, verbose=False)
quiet_loguru()

print("Discovered PTAs and pulsar counts:")
for pta, files in file_data.items():
    print(f"  {pta:<15s} -> {len(files):>3d} pulsars")

Discovered PTAs and pulsar counts:
  EPTA dr2        ->  42 pulsars
  NANOGrav 9y     ->  37 pulsars
  PPTA dr1dr2     ->  18 pulsars


`get_pulsar_names_from_file_data` does **coordinate-based** matching across PTAs: it parses each `.par` file's RAJ/DECJ (or ELONG/ELAT), normalises to a canonical name, and returns the de-duplicated list of pulsars that appear in *any* PTA. This is what you want when one PTA labels a pulsar `J1853+1303` and another spells the J2000 name slightly differently or uses a `B`-name.

In [3]:
pulsar_names = get_pulsar_names_from_file_data(file_data)
quiet_loguru()
print(f"Coordinate-matched pulsars across all PTAs: {len(pulsar_names)}")
print("First few:", pulsar_names[:8])

2026-04-22 08:22:18.719 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:18.722 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:18.724 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:18.727 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:18.729 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


...

2026-04-22 08:22:18.798 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:18.800 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:18.805 | WARNING  | metapulsar.position_helpers:_extract_equatorial_coordinates_optimized:499 - Missing PMRA/PMDEC or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:18.811 | WARNING  | metapulsar.position_helpers:_extract_equatorial_coordinates_optimized:499 - Missing PMRA/PMDEC or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


Coordinate-matched pulsars across all PTAs: 63
First few: ['J0030+0451', 'J0034-0534', 'J0218+4232', 'J0610-2100', 'J0613-0200', 'J0621+1002', 'J0751+1807', 'J0900-3144']


`pta_summary` is the canonical "is my data healthy" diagnostic. It parses every `.par` file end-to-end (so it is the slowest discovery call, ~5 s for IPTA-DR2) and prints a per-pulsar breakdown of which PTAs observe it, the timing-model timespan and TOA count from each, and the picked reference PTA. This is the table you look at when deciding what `PULSAR_SUBSET` to pin for an analysis.

In [4]:
pta_summary(file_data)
quiet_loguru()

Quickly processing PTA files...


Found 63 pulsars:



J0030+0451
- EPTA dr2: 5504 days (15.1 years, 908 TOAs) -- Reference PTA
- NANOGrav 9y: 3204 days (8.8 years, 2455 TOAs)



J0034-0534
- EPTA dr2: 4935 days (13.5 years, 284 TOAs) -- Reference PTA

J0218+4232
- EPTA dr2: 6416 days (17.6 years, 1206 TOAs) -- Reference PTA



J0610-2100
- EPTA dr2: 2521 days (6.9 years, 1034 TOAs) -- Reference PTA



...

J1614-2230
- NANOGrav 9y: 1861 days (5.1 years, 7323 TOAs) -- Reference PTA

J1741+1351
- NANOGrav 9y: 1552 days (4.2 years, 1588 TOAs) -- Reference PTA

J1747-4036
- NANOGrav 9y: 609 days (1.7 years, 2771 TOAs) -- Reference PTA

J1832-0836
- NANOGrav 9y: 231 days (0.6 years, 1131 TOAs) -- Reference PTA

J1903+0327
- NANOGrav 9y: 1456 days (4.0 years, 1802 TOAs) -- Reference PTA



J1923+2515
- NANOGrav 9y: 803 days (2.2 years, 920 TOAs) -- Reference PTA

J1944+0907
- NANOGrav 9y: 2086 days (5.7 years, 1696 TOAs) -- Reference PTA

J1949+3106
- NANOGrav 9y: 455 days (1.2 years, 1409 TOAs) -- Reference PTA

J2017+0603
- NANOGrav 9y: 609 days (1.7 years, 1565 TOAs) -- Reference PTA

J2043+1711
- NANOGrav 9y: 834 days (2.3 years, 1382 TOAs) -- Reference PTA



J2214+3000
- NANOGrav 9y: 755 days (2.1 years, 2514 TOAs) -- Reference PTA

J2302+4442
- NANOGrav 9y: 614 days (1.7 years, 3037 TOAs) -- Reference PTA

J0437-4715
- PPTA dr1dr2: 2188 days (6.0 years, 2039 TOAs) -- Reference PTA

J0711-6830
- PPTA dr1dr2: 2188 days (6.0 years, 398 TOAs) -- Reference PTA



J1045-4509
- PPTA dr1dr2: 2188 days (6.0 years, 396 TOAs) -- Reference PTA

J1603-7202
- PPTA dr1dr2: 2188 days (6.0 years, 346 TOAs) -- Reference PTA



J2129-5721
- PPTA dr1dr2: 1972 days (5.4 years, 274 TOAs) -- Reference PTA



## Step 2 -- Filter to `PULSAR_SUBSET` and focus on `TARGET`

`filter_file_data_by_pulsars` shrinks `file_data` to just the pulsars in `PULSAR_SUBSET`. We then narrow further to a single pulsar (`TARGET = J1853+1303`) so the heavy `create_metapulsar` build stays under a minute on a laptop.

In [5]:
filtered_data = filter_file_data_by_pulsars(file_data, PULSAR_SUBSET)
quiet_loguru()

for pta, files in filtered_data.items():
    matched = sorted({Path(f["par"]).name for f in files})
    print(f"{pta:<15s} -> {matched}")

single_pulsar_data = {
    pta: [
        f
        for f in files
        if TARGET in Path(f["par"]).name or TARGET in Path(f["tim"]).name
    ]
    for pta, files in filtered_data.items()
}
single_pulsar_data = {pta: files for pta, files in single_pulsar_data.items() if files}
print(f"\nPTAs available for {TARGET}: {list(single_pulsar_data.keys())}")

2026-04-22 08:22:24.904 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:24.907 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:24.909 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:24.912 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:24.915 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


...

2026-04-22 08:22:25.243 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:25.248 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:25.260 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:25.265 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


EPTA dr2        -> ['J1853+1303.par', 'J1955+2908.par']
NANOGrav 9y     -> ['B1953+29_NANOGrav_9yv1.gls.par', 'J1853+1303_NANOGrav_9yv1.gls.par']

PTAs available for J1853+1303: ['EPTA dr2', 'NANOGrav 9y']


## Step 3 -- Build a consistent MetaPulsar (auto reference PTA)

When `reference_pta=None` (the default), the factory picks the PTA with the longest timespan. We pre-stage the *original* per-PTA `.par` files into `./parfiles/` (so we can diff them against the rewritten consistent ones in step 5), then build the consistent MetaPulsar with `parfile_output_dir="./parfiles"`.

The bulk of the time is `parameter_manager.make_parfiles_consistent()` (parsing every par file into PINT, copying mergeable parameters, re-emitting harmonised par files). With only EPTA + NANOGrav for `J1853+1303` and ~1.5k TOAs, this is ~30-60 s on a laptop.

In [6]:
import shutil

from metapulsar import create_metapulsar

parfiles_dir = Path("./parfiles").resolve()
parfiles_dir.mkdir(exist_ok=True)

for pta, files in single_pulsar_data.items():
    src = Path(files[0]["par"])
    dest = parfiles_dir / f"{TARGET}_original_{pta}.par"
    shutil.copy(src, dest)
    print(f"  copied original  {pta:<15s} -> {dest.name}")

mp_consistent = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="consistent",
    combine_components=["astrometry", "spindown", "binary", "dispersion"],
    add_dm_derivatives=True,
    parfile_output_dir=str(parfiles_dir),
)
quiet_loguru()

print()
print(f"Name              : {mp_consistent.name}")
print(f"Strategy          : {mp_consistent.combination_strategy}")
print(f"PTAs combined     : {list(mp_consistent._pulsars.keys())}")
print(f"Reference PTA     : {list(mp_consistent._pulsars.keys())[0]}")
print(f"Components merged : {mp_consistent.combine_components}")
print(f"Total TOAs        : {len(mp_consistent.toas)}")
print(f"Fit parameters    : {len(mp_consistent.fitpars)}")

  copied original  EPTA dr2        -> J1853+1303_original_EPTA dr2.par
  copied original  NANOGrav 9y     -> J1853+1303_original_NANOGrav 9y.par
2026-04-22 08:22:25.321 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:25.324 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:22:25.515 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.


2026-04-22 08:22:25.520 | WARNING  | pint.models.model_builder:__call__:224 - UNITS is not specified. Assuming TDB...


2026-04-22 08:22:25.608 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.


...

DMASSPLANET4 (M inf                       0                         0             -inf          N


DMASSPLANET5 (M inf                       0                         0             -inf          N


DMASSPLANET6 (M inf                       0                         0             -inf          N


DMASSPLANET7 (M inf                       0                         0             -inf          N


DMASSPLANET8 (M 0                         0                         0             0             N


## Step 4 -- Force a different reference PTA

The reference PTA contributes the *values* of every merged parameter -- it is therefore the only PTA whose `.par` is preserved verbatim. Forcing a different reference is mostly a sensitivity knob: a well-constrained pulsar should be statistically indistinguishable across reference choices. We rebuild the same pulsar with NANOGrav 9y as the reference (when present) and write to a separate `parfiles_ngref/` directory.

In [7]:
FORCED_REF = (
    "NANOGrav 9y"
    if "NANOGrav 9y" in single_pulsar_data
    else next(iter(single_pulsar_data))
)

mp_consistent_ngref = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="consistent",
    reference_pta=FORCED_REF,
    parfile_output_dir="./parfiles_ngref",
)
quiet_loguru()

print(f"Forced reference PTA   : {FORCED_REF}")
print(f"PTAs (reference first) : {list(mp_consistent_ngref._pulsars.keys())}")
print(f"Fit parameter count    : {len(mp_consistent_ngref.fitpars)}")

2026-04-22 08:22:49.342 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


[textOutput.C:308] Notice: Parameter uncertainties NOT multiplied by sqrt(red. chisq)


2026-04-22 08:22:49.345 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


Jump 1 (                -sys JBO.DFB.1520 0 1): 0 0 Y


Jump 2 (                -sys NRT.BON.1600 0 1): 0 0 Y


...

DMASSPLANET9 (M inf                       0                         0             -inf          N


NE_SW (cm^-3)   4                         4                         0             0             N


DM_SERIES       TAYLOR                   


---------------------------------------------------------------------------------------------------


[textOutput.C:308] Notice: Parameter uncertainties NOT multiplied by sqrt(red. chisq)


## Step 5 -- Diff the original vs. consistent par files

This is where the consistent strategy stops being abstract. For each non-reference PTA we side-by-side print the merged-component lines from the *original* par file (the one we copied from the IPTA-DR2 release) and from the *consistent* par file (rewritten by `parameter_manager.make_parfiles_consistent`). The values in the consistent file should now match the reference PTA's original values exactly; everything outside the merged components (JUMPs, EFAC/EQUAD per backend, DMX, ...) is left alone.

In [8]:
import re

MERGED_PAR_KEYS = {
    "astrometry": ["RAJ", "DECJ", "PMRA", "PMDEC", "PX", "POSEPOCH", "ELONG", "ELAT"],
    "spindown": ["F0", "F1", "F2", "PEPOCH"],
    "binary": ["BINARY", "PB", "A1", "OM", "T0", "ECC", "EPS1", "EPS2", "TASC", "M2", "SINI"],
    "dispersion": ["DM", "DM1", "DM2", "DMEPOCH"],
}
FLAT_KEYS = sorted({k for ks in MERGED_PAR_KEYS.values() for k in ks})


def parfile_value_map(par_text: str) -> dict:
    out = {}
    for line in par_text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        tokens = re.split(r"\s+", line)
        key = tokens[0]
        if key in FLAT_KEYS and len(tokens) >= 2:
            out[key] = tokens[1]
    return out


ref_pta = list(mp_consistent._pulsars.keys())[0]
non_ref_ptas = [pta for pta in mp_consistent._pulsars.keys() if pta != ref_pta]

print(f"Reference PTA: {ref_pta}\n")
for pta in non_ref_ptas:
    orig = next(parfiles_dir.glob(f"*{TARGET}_original_{pta}.par"), None)
    cons = next(parfiles_dir.glob(f"*{TARGET}_consistent_{pta}.par"), None)
    if orig is None or cons is None:
        print(f"[skip] {pta}: missing original or consistent file (orig={orig}, cons={cons})")
        continue
    orig_map = parfile_value_map(orig.read_text())
    cons_map = parfile_value_map(cons.read_text())

    print(f"=== {pta}: merged-component keys, original -> consistent ===")
    keys = sorted(set(orig_map) | set(cons_map))
    for key in keys:
        o = orig_map.get(key, "--")
        c = cons_map.get(key, "--")
        marker = "" if o == c else "   <-- changed"
        print(f"  {key:<8s}  orig={o:<28s}  cons={c:<28s}{marker}")
    print()

Derived parameters:


P0 (s)      = 0.00409179738307904       3.2945e-17   


P1          = 8.71644322717868e-21      3.6353e-25   


tau_c (Myr) = 7442.8


bs (G)      = 1.9111e+08


...

Dilate freq?           Yes (tempo2)


Electron density (1AU) 4


Solar system ephem     DE421


Time scale             TT(BIPM)


Binary model           DD


## Step 6 -- Persist the file-data dict for notebook 03

MetaPulsar objects hold open file handles / thread locks and are therefore not picklable, which means we cannot pass them directly through IPython's `%store`. What we *can* persist is the plain `dict` of `{pta: [{par, tim, ...}, ...]}` entries that drives `create_metapulsar`. Notebook 03 picks that dict up and rebuilds the consistent MetaPulsar in a single call.

We deliberately do **not** rebuild the second pulsar (`B1953+29`) here. The recipe is identical to step 3; running it once for `J1853+1303` already exercises the full consistent path. For a production batch, see `create_all_metapulsars(file_data, combination_strategy="consistent")` in `examples/notebooks/using_metapulsar.ipynb`.

In [9]:
FILE_DATA_REGISTRY = {TARGET: single_pulsar_data}
TUTORIAL_TARGET = TARGET
%store FILE_DATA_REGISTRY
%store TUTORIAL_TARGET
print("Stored FILE_DATA_REGISTRY and TUTORIAL_TARGET for 03_consistency_checks.ipynb.")
print("Notebook 03 will rebuild the consistent MetaPulsar from this dict.")

Stored 'FILE_DATA_REGISTRY' (dict)
Stored 'TUTORIAL_TARGET' (str)
Stored FILE_DATA_REGISTRY and TUTORIAL_TARGET for 03_consistency_checks.ipynb.
Notebook 03 will rebuild the consistent MetaPulsar from this dict.
